In [1]:
# Cell 1: Import necessary libraries
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from mpl_toolkits.mplot3d import Axes3D  # Import for 3D plotting
from IPython.display import display, HTML

In [2]:
# Cell 2: Particle Swarm Optimization class implementation
class ParticleSwarmOptimization:
    def __init__(self, objective_function, dimensions, bounds, num_particles=30, 
                 max_iterations=500, w=0.7, c1=1.5, c2=1.5):
        """
        Initialize the PSO algorithm
        
        Parameters:
        - objective_function: the function to minimize
        - dimensions: number of dimensions in the search space
        - bounds: tuple of (min_bound, max_bound) for each dimension
        - num_particles: population size
        - max_iterations: maximum number of iterations
        - w: inertia weight
        - c1: cognitive parameter (personal best)
        - c2: social parameter (global best)
        """
        self.objective_function = objective_function
        self.dimensions = dimensions
        self.bounds = bounds
        self.num_particles = num_particles
        self.max_iterations = max_iterations
        self.w = w
        self.c1 = c1
        self.c2 = c2
        
        # Initialize particles and velocities
        self.particles = np.random.uniform(bounds[0], bounds[1], 
                                          (num_particles, dimensions))
        # Smaller initial velocities for better control
        self.velocities = np.random.uniform(-0.5, 0.5, (num_particles, dimensions))
        
        # Initialize personal and global best
        self.personal_best_positions = self.particles.copy()
        self.personal_best_values = np.array([objective_function(p) for p in self.particles])
        
        global_best_idx = np.argmin(self.personal_best_values)
        self.global_best_position = self.personal_best_positions[global_best_idx].copy()
        self.global_best_value = self.personal_best_values[global_best_idx]
        
        # For tracking history
        self.position_history = [self.particles.copy()]
        self.global_best_history = [self.global_best_value]
        
    def update(self):
        """Run one iteration of the PSO algorithm"""
        # Update velocities
        r1, r2 = np.random.random((self.num_particles, self.dimensions)), np.random.random((self.num_particles, self.dimensions))
        
        cognitive_component = self.c1 * r1 * (self.personal_best_positions - self.particles)
        social_component = self.c2 * r2 * (self.global_best_position - self.particles)
        
        self.velocities = self.w * self.velocities + cognitive_component + social_component
        
        # Limit velocity to prevent explosive behavior
        max_velocity = 0.2 * (self.bounds[1] - self.bounds[0])
        self.velocities = np.clip(self.velocities, -max_velocity, max_velocity)
        
        # Update positions
        self.particles = self.particles + self.velocities
        
        # Clamp positions to bounds
        self.particles = np.clip(self.particles, self.bounds[0], self.bounds[1])
        
        # Update personal best
        current_values = np.array([self.objective_function(p) for p in self.particles])
        
        # Update personal best when current value is better (lower for minimization)
        mask = current_values < self.personal_best_values
        self.personal_best_positions[mask] = self.particles[mask].copy()
        self.personal_best_values[mask] = current_values[mask]
        
        # Update global best
        min_idx = np.argmin(self.personal_best_values)
        if self.personal_best_values[min_idx] < self.global_best_value:
            self.global_best_position = self.personal_best_positions[min_idx].copy()
            self.global_best_value = self.personal_best_values[min_idx]
        
        # Save history for visualization
        self.position_history.append(self.particles.copy())
        self.global_best_history.append(self.global_best_value)
    
    def optimize(self, visualize=False, visualization_dimensions=(0, 1)):
        """
        Run the PSO algorithm for max_iterations
        
        Parameters:
        - visualize: if True, create an animation of the optimization process
        - visualization_dimensions: which dimensions to visualize if visualize=True
                                   (default: first two dimensions)
        
        Returns:
        - global_best_position: the best solution found
        - global_best_value: the value of the best solution
        """
        for i in range(self.max_iterations):
            self.update()
        
        if visualize and self.dimensions >= 2:
            self._visualize_optimization(visualization_dimensions)
        
        return self.global_best_position, self.global_best_value
    
    def _visualize_optimization(self, dims=(0, 1)):
        """Create an animation of the optimization process"""
        dim1, dim2 = dims
        
        if self.dimensions <= max(dims):
            raise ValueError("Visualization dimensions must be valid for the problem dimensions")
        
        fig, ax = plt.subplots(1, 2, figsize=(15, 6))
        
        # For the particle movement plot
        scatter = ax[0].scatter(
            self.position_history[0][:, dim1], 
            self.position_history[0][:, dim2], 
            c='blue', alpha=0.5
        )
        
        global_best_scatter = ax[0].scatter(
            self.global_best_position[dim1], 
            self.global_best_position[dim2], 
            c='red', s=100, marker='*'
        )
        
        ax[0].set_xlim(self.bounds[0] - 0.1, self.bounds[1] + 0.1)
        ax[0].set_ylim(self.bounds[0] - 0.1, self.bounds[1] + 0.1)
        ax[0].set_title("Particle Positions")
        ax[0].set_xlabel(f"Dimension {dim1}")
        ax[0].set_ylabel(f"Dimension {dim2}")
        
        # For the convergence plot
        ax[1].plot(self.global_best_history, 'b-', lw=2)
        ax[1].set_xlim(0, self.max_iterations)
        ax[1].set_ylim(min(min(self.global_best_history) * 0.9, 0.1), 
                       max(max(self.global_best_history[:10]) * 1.1, 1.0))
        ax[1].set_title("Convergence")
        ax[1].set_xlabel("Iteration")
        ax[1].set_ylabel("Global Best Value")
        
        iteration_text = ax[0].text(
            0.02, 0.95, f'Iteration: {self.max_iterations}', 
            transform=ax[0].transAxes, fontsize=12
        )
        
        # Plot final state at the end of optimization
        plt.tight_layout()
        plt.show()
        
    def animate_optimization(self, dims=(0, 1)):
        """Create an animation of the optimization process - separate function for Jupyter"""
        dim1, dim2 = dims
        
        if self.dimensions <= max(dims):
            raise ValueError("Visualization dimensions must be valid for the problem dimensions")
            
        fig_anim, axes = plt.subplots(1, 2, figsize=(15, 6))
        
        # Set up the first plot - particle positions
        particles_scatter = axes[0].scatter(
            [], [], c='blue', alpha=0.5
        )
        best_scatter = axes[0].scatter(
            [], [], c='red', s=100, marker='*'
        )
        
        axes[0].set_xlim(self.bounds[0] - 0.1, self.bounds[1] + 0.1)
        axes[0].set_ylim(self.bounds[0] - 0.1, self.bounds[1] + 0.1)
        axes[0].set_title("Particle Positions")
        axes[0].set_xlabel(f"Dimension {dim1}")
        axes[0].set_ylabel(f"Dimension {dim2}")
        
        # Set up the second plot - convergence
        line, = axes[1].plot([], [], 'b-', lw=2)
        axes[1].set_xlim(0, self.max_iterations)
        axes[1].set_ylim(min(min(self.global_best_history) * 0.9, 0.1), 
                       max(max(self.global_best_history[:10]) * 1.1, 1.0))
        axes[1].set_title("Convergence")
        axes[1].set_xlabel("Iteration")
        axes[1].set_ylabel("Global Best Value")
        
        iter_text = axes[0].text(
            0.02, 0.95, '', transform=axes[0].transAxes, fontsize=12
        )
        
        def init():
            particles_scatter.set_offsets(np.column_stack((
                self.position_history[0][:, dim1], 
                self.position_history[0][:, dim2]
            )))
            
            best_idx = np.argmin([self.objective_function(p) for p in self.position_history[0]])
            best_position = self.position_history[0][best_idx]
            best_scatter.set_offsets(np.array([
                best_position[dim1],
                best_position[dim2]
            ]))
            
            line.set_data([], [])
            iter_text.set_text('Iteration: 0')
            return particles_scatter, best_scatter, line, iter_text
        
        def update(frame):
            particles_scatter.set_offsets(np.column_stack((
                self.position_history[frame][:, dim1], 
                self.position_history[frame][:, dim2]
            )))
            
            # Use the actual global best for this iteration
            best_pos = self.position_history[frame][
                np.argmin([self.objective_function(p) for p in self.position_history[frame]])
            ]
            
            best_scatter.set_offsets(np.array([
                best_pos[dim1],
                best_pos[dim2]
            ]))
            
            line.set_data(range(frame+1), self.global_best_history[:frame+1])
            iter_text.set_text(f'Iteration: {frame}')
            return particles_scatter, best_scatter, line, iter_text
        
        ani = FuncAnimation(
            fig_anim, update, frames=range(len(self.position_history)),
            init_func=init, blit=True, interval=100
        )
        
        plt.tight_layout()
        return ani



In [3]:
# Cell 3: Visualization utilities (separated from the PSO class for clarity)
def visualize_optimization_results(pso, objective_function, bounds):
    """
    Create visualization of the optimization results
    
    Parameters:
    - pso: ParticleSwarmOptimization instance with optimization history
    - objective_function: the function that was optimized
    - bounds: tuple of (min_bound, max_bound) for the search space
    """
    best_position = pso.global_best_position
    best_value = pso.global_best_value
    
    # Create a grid for visualization
    x = np.linspace(bounds[0], bounds[1], 100)
    y = np.linspace(bounds[0], bounds[1], 100)
    X, Y = np.meshgrid(x, y)
    Z = np.zeros_like(X)
    
    # Calculate function values for 2D visualization
    for i in range(len(x)):
        for j in range(len(y)):
            Z[j, i] = objective_function(np.array([X[j, i], Y[j, i]]))
    
    # Create figure with subplots - 2D contour and 3D surface
    fig = plt.figure(figsize=(16, 8))
    
    # 2D Contour plot
    ax1 = fig.add_subplot(1, 2, 1)
    
    # Use log scale for contour levels
    contour_levels = np.logspace(-1, 3, 20)
    contour = ax1.contour(X, Y, Z, levels=contour_levels, cmap='viridis', alpha=0.6)
    plt.colorbar(contour, ax=ax1, label='Function Value')
    
    # Plot particle history with focused trajectories
    history_positions = np.array(pso.position_history)
    
    # Sample only a few particles to avoid overcrowding
    sample_size = min(10, pso.num_particles)
    particle_indices = np.random.choice(pso.num_particles, sample_size, replace=False)
    
    for i in particle_indices:
        particle_history = history_positions[:, i, :]
        ax1.plot(particle_history[:, 0], particle_history[:, 1], 'b-', alpha=0.4)
    
    # Mark final positions and best position
    final_positions = pso.position_history[-1]
    ax1.scatter(final_positions[:, 0], final_positions[:, 1], c='blue', alpha=0.5, label='Final Positions')
    ax1.scatter(best_position[0], best_position[1], c='red', s=100, marker='*', label='Best Position')
    
    # Add true minimum if it's the Rosenbrock function
    ax1.scatter(1, 1, c='green', s=100, marker='o', label='True Minimum (1,1)')
    
    ax1.set_title('2D View of Function with PSO Trajectories')
    ax1.set_xlabel('x')
    ax1.set_ylabel('y')
    ax1.grid(True)
    ax1.legend()
    
    # 3D Surface plot
    ax2 = fig.add_subplot(1, 2, 2, projection='3d')
    
    # Use log scale for better visualization
    Z_log = np.log10(Z + 1)  # Add 1 to avoid log(0)
    surface = ax2.plot_surface(X, Y, Z_log, cmap='viridis', alpha=0.8, 
                               linewidth=0, antialiased=True)
    
    # Calculate z-values for final positions
    final_z_values = np.array([np.log10(objective_function(np.array([p[0], p[1]])) + 1) 
                               for p in final_positions])
    
    # Plot final particle positions
    ax2.scatter(final_positions[:, 0], final_positions[:, 1], final_z_values,
                c='blue', alpha=0.5, label='Final Positions')
    
    # Plot best position
    best_z = np.log10(objective_function(np.array([best_position[0], best_position[1]])) + 1)
    ax2.scatter(best_position[0], best_position[1], best_z,
                c='red', s=100, marker='*', label='Best Position')
    
    # Plot true minimum (for Rosenbrock)
    true_min_z = np.log10(objective_function(np.array([1, 1])) + 1)
    ax2.scatter(1, 1, true_min_z, c='green', s=100, marker='o', label='True Minimum')
    
    # Add color bar
    cbar = fig.colorbar(surface, ax=ax2, shrink=0.7, aspect=10, pad=0.1)
    cbar.set_label('Log10(Function Value)')
    
    # Set labels and title
    ax2.set_xlabel('x')
    ax2.set_ylabel('y')
    ax2.set_zlabel('Log10(Function Value)')
    ax2.set_title('3D View of Function')
    
    # Set optimal viewing angle
    ax2.view_init(elev=30, azim=45)
    
    plt.tight_layout()
    plt.show()
    
def create_detailed_3d_view(pso, objective_function, bounds):
    """Create a more detailed 3D visualization with particle trajectories"""
    # Create a grid for visualization
    x = np.linspace(bounds[0], bounds[1], 100)
    y = np.linspace(bounds[0], bounds[1], 100)
    X, Y = np.meshgrid(x, y)
    Z = np.zeros_like(X)
    
    # Calculate function values
    for i in range(len(x)):
        for j in range(len(y)):
            Z[j, i] = objective_function(np.array([X[j, i], Y[j, i]]))
    
    # Use log scale for better visualization
    Z_log = np.log10(Z + 1)  # Add 1 to avoid log(0)
    
    best_position = pso.global_best_position
    
    # Create figure
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')
    
    # Create 3D surface
    surface = ax.plot_surface(X, Y, Z_log, cmap='viridis', alpha=0.7, 
                              linewidth=0, antialiased=True)
    
    # Get history and sample particles
    history_positions = np.array(pso.position_history)
    sample_size = min(10, pso.num_particles)
    particle_indices = np.random.choice(pso.num_particles, sample_size, replace=False)
    
    # Plot trajectories of selected particles in 3D
    for i in particle_indices:
        particle_history = history_positions[:, i, :]
        # Calculate z values for the trajectory
        trajectory_z = np.array([np.log10(objective_function(np.array([p[0], p[1]])) + 1) 
                               for p in particle_history])
        ax.plot(particle_history[:, 0], particle_history[:, 1], trajectory_z, 'b-', alpha=0.4)
    
    # Plot final positions
    final_positions = pso.position_history[-1]
    final_z_values = np.array([np.log10(objective_function(np.array([p[0], p[1]])) + 1) 
                               for p in final_positions])
    
    ax.scatter(final_positions[:, 0], final_positions[:, 1], final_z_values,
               c='blue', alpha=0.5, label='Final Positions')
    
    # Plot best position with larger marker
    best_z = np.log10(objective_function(np.array([best_position[0], best_position[1]])) + 1)
    ax.scatter(best_position[0], best_position[1], best_z,
               c='red', s=100, marker='*', label='Best Position')
    
    # Plot true minimum (for Rosenbrock)
    true_min_z = np.log10(objective_function(np.array([1, 1])) + 1)
    ax.scatter(1, 1, true_min_z, c='green', s=100, marker='o', label='True Minimum (1,1)')
    
    # Add color bar
    cbar = fig.colorbar(surface, ax=ax, shrink=0.7)
    cbar.set_label('Log10(Function Value)')
    
    # Set labels and title
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_zlabel('Log10(Function Value)')
    ax.set_title('3D Visualization with PSO Trajectories')
    
    # Set view angle for best visualization
    ax.view_init(elev=35, azim=45)
    ax.set_zlim(0, np.max(Z_log))
    
    plt.tight_layout()
    plt.show()

In [4]:
# Cell 4: Define the Rosenbrock function
def rosenbrock(x):
    """
    Rosenbrock function (banana function)
    Global minimum at (1, 1, ..., 1)
    """
    return sum(100.0 * (x[1:] - x[:-1]**2)**2 + (1 - x[:-1])**2)

In [5]:
# Cell 5: Create a demo for optimizing the Rosenbrock function
def optimize_rosenbrock():
    """Demo optimizing the Rosenbrock function with PSO"""
    dimensions = 2
    bounds = (-2, 2)
    
    print("Optimizing Rosenbrock Function with PSO")
    print("-" * 50)
    
    # Initialize PSO
    pso = ParticleSwarmOptimization(
        objective_function=rosenbrock,
        dimensions=dimensions,
        bounds=bounds,
        num_particles=30,
        max_iterations=100,
        w=0.7,  # Slightly higher inertia for better exploration
        c1=1.5, # Balanced cognitive component
        c2=1.5  # Balanced social component
    )
    
    # Run optimization
    best_position, best_value = pso.optimize()
    
    # Display results
    print(f"Best solution found: {best_position}")
    print(f"Best value: {best_value}")
    print(f"True global minimum: [1, 1] with value 0")
    
    # Show basic visualization (2D + convergence)
    pso._visualize_optimization()
    
    # Show detailed visualizations
    visualize_optimization_results(pso, rosenbrock, bounds)
    create_detailed_3d_view(pso, rosenbrock, bounds)
    
    # Create animation
    print("Creating animation (might take a moment)...")
    animation = pso.animate_optimization()
    
    # Display animation in the notebook
    return animation


In [6]:

# Cell 6: Run the Rosenbrock optimization demo
# Uncomment to run:

animation = optimize_rosenbrock()
# Note: To display the animation in Jupyter, you might need:
from IPython.display import HTML
HTML(animation.to_jshtml())

Optimizing Rosenbrock Function with PSO
--------------------------------------------------
Best solution found: [1.00060813 1.00127166]
Best value: 6.726488028852407e-07
True global minimum: [1, 1] with value 0
Creating animation (might take a moment)...
